In [0]:
use catalog practise;
use schema task;

In [0]:
select*from car_table

In [0]:
GRANT USE SCHEMA ON SCHEMA practise.task1 TO `nimonisha@gmail.com`;

In [0]:
GRANT SELECT ON TABLE practise.task1.car_table TO `nimonisha@gmail.com`;

In [0]:
select * from practise.task1.car_table

In [0]:
use catalog employee;
use schema bronze;

In [0]:
CREATE OR REPLACE TABLE employee.bronze.employees_raw
USING DELTA
AS
SELECT
  id AS employee_id,
  concat('Emp_', id) AS full_name,
  floor(rand()*30 + 22) AS age,
  element_at(array('Male','Female'), cast(rand()*2+1 AS INT)) AS gender,
  element_at(array('IT','HR','Finance','Sales','Marketing'), cast(rand()*5+1 AS INT)) AS department,
  element_at(array('Engineer','Analyst','Manager','Lead','Consultant'), cast(rand()*5+1 AS INT)) AS job_title,
  floor(rand()*15 + 1) AS experience_years,
  element_at(array('Bachelors','Masters','MBA','PhD'), cast(rand()*4+1 AS INT)) AS education_level,
  element_at(array('Chennai','Bangalore','Hyderabad','Pune','Mumbai'), cast(rand()*5+1 AS INT)) AS location,
  current_timestamp() AS created_ts
FROM range(20000);

In [0]:
select * from employee.bronze.employees_raw

In [0]:
CREATE OR REPLACE TABLE employee.bronze.salaries_raw
USING DELTA
AS
SELECT
  employee_id,
  floor(rand()*40000 + 30000) AS base_salary,
  floor(rand()*10000) AS bonus,
  floor(rand()*5000) AS allowances,
  round(rand()*20,2) AS tax_percent,
  current_date() AS effective_date
FROM employee.bronze.employees_raw;

In [0]:
%python
df = spark.read.table("employee.bronze.employees_raw")
df.show()

In [0]:
select * from employee.bronze.salaries_raw

In [0]:
CREATE OR REPLACE TABLE employee.bronze.attendance_raw
USING DELTA
AS
SELECT
  e.employee_id,
  date_sub(current_date(), CAST(d.id AS INT)) AS work_date,
  round(rand()*9,2) AS hours_worked,
  element_at(array('Present','Absent','WFH'), cast(rand()*3+1 AS INT)) AS status
FROM employee.bronze.employees_raw e
CROSS JOIN range(30) d;

In [0]:
select * from  employee.bronze.attendance_raw


In [0]:
CREATE OR REPLACE TABLE employee.bronze.leaves_raw
USING DELTA
AS
SELECT
  employee_id,
  element_at(array('Sick','Casual','Vacation','Maternity'), cast(rand()*4+1 AS INT)) AS leave_type,
  floor(rand()*10) AS leave_days,
  date_sub(current_date(), cast(rand()*100 AS INT)) AS applied_date
FROM employee.bronze.employees_raw
WHERE rand() < 0.4;

In [0]:
select * from employee.bronze.leaves_raw

In [0]:
CREATE OR REPLACE TABLE employee.bronze.performance_raw
USING DELTA
AS
SELECT
  employee_id,
  2025 AS review_year,
  round(rand()*5,1) AS rating,
  element_at(array('Excellent','Good','Average','Poor'), cast(rand()*4+1 AS INT)) AS grade,
  element_at(array('Yes','No'), cast(rand()*2+1 AS INT)) AS promotion_flag,
  concat('Mgr_', cast(rand()*50 AS INT)) AS reviewer
FROM employee.bronze.employees_raw;

In [0]:
select * from employee.bronze.performance_raw

In [0]:
CREATE OR REPLACE TABLE employee.bronze.training_raw
USING DELTA
AS
SELECT
  e.employee_id,
  element_at(array('Python','SQL','Spark','Leadership','AI'), cast(rand()*5+1 AS INT)) AS course_name,
  floor(rand()*40 + 5) AS hours_spent,
  round(rand()*100,1) AS score,
  element_at(array('Completed','In Progress'), cast(rand()*2+1 AS INT)) AS status
FROM employee.bronze.employees_raw e
CROSS JOIN range(5);

In [0]:
select * from employee.bronze.training_raw

In [0]:
CREATE OR REPLACE TABLE employee.bronze.recruitment_raw
USING DELTA
AS
SELECT
  employee_id,
  date_sub(current_date(), cast(rand()*2000 AS INT)) AS hire_date,
  element_at(array('LinkedIn','Referral','Agency','Campus'), cast(rand()*4+1 AS INT)) AS source,
  concat('Rec_', cast(rand()*100 AS INT)) AS recruiter_name,
  floor(rand()*50000) AS hiring_cost
FROM employee.bronze.employees_raw;

In [0]:
select * from employee.bronze.recruitment_raw

In [0]:
CREATE OR REPLACE TABLE employee.bronze.department_budget_raw
USING DELTA
AS
SELECT
  department,
  2025 AS fiscal_year,
  floor(rand()*2000000 + 500000) AS budget,
  floor(rand()*1500000) AS expenses,
  concat('Head_', cast(rand()*20 AS INT)) AS department_head
FROM employee.bronze.employees_raw
GROUP BY department;

In [0]:
select * from employee.bronze.department_budget_raw